# silver_claims

Promotes claims from two bronze sources into a single reconciled silver table. Same pattern as silver_policies: UNION, source priority, validation, reject quarantine.

Claims is simpler than policies in one way: the only FK is policy_id, which must exist in silver_policies. No carrier or agent FK to check here. But that FK check is meaningful, a claim against a policy that failed silver validation would itself be invalid, and we don't want it in reporting.

One thing worth knowing about this data: the generator filtered out dirty policies when building claims (it only wrote claims against ACTIVE or EXPIRED policies with valid premiums and dates). So claims should be relatively clean. The rejects table will likely be sparse, but the infrastructure is here because in production you'd never assume clean data without checking.

## FK validation

- `policy_id` must exist in `silver_policies`

We validate against silver_policies, not bronze_policies. A claim against a policy that failed silver validation is itself suspect.

## Source and output

- `bronze_citadel_mga.bronze_claims` (CSV, 2,000 rows)
- `bronze_citadel_mga.bronze_claims_api` (API, 2,000 rows)

Outputs:
- `silver_citadel_mga.silver_claims`
- `silver_citadel_mga.silver_claims_rejects`

## Source priority

CSV wins on conflicts. API-only claims are included.

## What's NOT here

No SCD2. Claims are immutable once closed — a closed claim doesn't change its amount or status retroactively in normal operations. If a claim gets reopened, that's a new event in IMS, not an update to the existing record.


In [1]:
from pyspark.sql.functions import lit, current_timestamp, col, when, concat_ws
from pyspark.sql.types import StringType
import uuid
from datetime import datetime

SILVER_BATCH_ID = f"silver_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

# Two bronze sources.
SOURCE_CSV = "bronze_citadel_mga.dbo.bronze_claims"
SOURCE_API = "bronze_citadel_mga.dbo.bronze_claims_api"

# Two silver outputs.
TARGET_TABLE  = "silver_claims"
REJECTS_TABLE = "silver_claims_rejects"

# FK reference — validate against silver_policies.
REF_POLICIES = "silver_citadel_mga.dbo.silver_policies"

# Valid claim statuses.
VALID_CLAIM_STATUSES = {"OPEN", "CLOSED", "LITIGATION", "DENIED"}

print(f"Silver batch ID: {SILVER_BATCH_ID}")
print(f"CSV source:      {SOURCE_CSV}")
print(f"API source:      {SOURCE_API}")
print(f"Target:          {TARGET_TABLE}")
print(f"Rejects:         {REJECTS_TABLE}")
print(f"FK ref policies: {REF_POLICIES}")

StatementMeta(, 8081f143-5a0a-4e32-8063-b016b7eb9547, 3, Finished, Available, Finished, False)

Silver batch ID: silver_20260603_014023_db062087
CSV source:      bronze_citadel_mga.dbo.bronze_claims
API source:      bronze_citadel_mga.dbo.bronze_claims_api
Target:          silver_claims
Rejects:         silver_claims_rejects
FK ref policies: silver_citadel_mga.dbo.silver_policies


In [2]:
csv_df = spark.read.table(SOURCE_CSV)
api_df = spark.read.table(SOURCE_API)

print(f"CSV source:  {csv_df.count()} rows, {len(csv_df.columns)} cols")
print(f"API source:  {api_df.count()} rows, {len(api_df.columns)} cols")
print()
print(f"CSV columns: {csv_df.columns}")
print()
print(f"API columns: {api_df.columns}")
print()

# Load FK reference set from silver_policies.
valid_policy_ids = set(
    row["policy_id"]
    for row in spark.read.table(REF_POLICIES).select("policy_id").collect()
)
print(f"Valid policy_ids loaded: {len(valid_policy_ids)}")
print()

# Check domain column mismatches.
csv_domain = set(csv_df.columns) - {"_ingestion_timestamp", "_source_file", "_batch_id", "_ingestion_method"}
api_domain  = set(api_df.columns) - {"_ingestion_timestamp", "_source_endpoint", "_batch_id", "_ingestion_method"}

only_in_csv = csv_domain - api_domain
only_in_api = api_domain - csv_domain

print(f"Domain cols only in CSV: {only_in_csv if only_in_csv else 'none'}")
print(f"Domain cols only in API: {only_in_api if only_in_api else 'none'}")

StatementMeta(, 8081f143-5a0a-4e32-8063-b016b7eb9547, 4, Finished, Available, Finished, False)

CSV source:  2000 rows, 11 cols
API source:  2000 rows, 11 cols

CSV columns: ['claim_id', 'policy_id', 'loss_date', 'report_date', 'claim_amount', 'claim_status', 'cause_of_loss', '_ingestion_timestamp', '_source_file', '_batch_id', '_ingestion_method']

API columns: ['cause_of_loss', 'claim_amount', 'claim_id', 'claim_status', 'loss_date', 'policy_id', 'report_date', '_ingestion_timestamp', '_source_endpoint', '_batch_id', '_ingestion_method']

Valid policy_ids loaded: 9709

Domain cols only in CSV: none
Domain cols only in API: none


In [3]:
csv_standardized = (
    csv_df
    .withColumnRenamed("_batch_id", "_bronze_batch_id")
    .withColumn("_source_system", lit("csv"))
    .drop("_ingestion_timestamp", "_source_file", "_ingestion_method")
)

api_standardized = (
    api_df
    .withColumnRenamed("_batch_id", "_bronze_batch_id")
    .withColumn("_source_system", lit("api"))
    .drop("_ingestion_timestamp", "_source_endpoint", "_ingestion_method")
)

unioned_df = csv_standardized.unionByName(api_standardized)

print(f"CSV rows after standardize:  {csv_standardized.count()}")
print(f"API rows after standardize:  {api_standardized.count()}")
print(f"Combined rows after UNION:   {unioned_df.count()}")
print()
print(f"Columns: {unioned_df.columns}")

StatementMeta(, 8081f143-5a0a-4e32-8063-b016b7eb9547, 5, Finished, Available, Finished, False)

CSV rows after standardize:  2000
API rows after standardize:  2000
Combined rows after UNION:   4000

Columns: ['claim_id', 'policy_id', 'loss_date', 'report_date', 'claim_amount', 'claim_status', 'cause_of_loss', '_bronze_batch_id', '_source_system']


In [4]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

priority_df = unioned_df.withColumn(
    "_priority",
    when(col("_source_system") == "csv", 1).otherwise(2)
)

window_spec = Window.partitionBy("claim_id").orderBy("_priority")

ranked_df = priority_df.withColumn("_rank", row_number().over(window_spec))

deduped_df = (
    ranked_df
    .filter(col("_rank") == 1)
    .drop("_priority", "_rank")
)

csv_count      = deduped_df.filter(col("_source_system") == "csv").count()
api_only_count = deduped_df.filter(col("_source_system") == "api").count()

print(f"Rows after deduplication:  {deduped_df.count()}")
print(f"  From CSV:                {csv_count}")
print(f"  From API only:           {api_only_count}")
print()
print(f"Columns: {deduped_df.columns}")

StatementMeta(, 8081f143-5a0a-4e32-8063-b016b7eb9547, 6, Finished, Available, Finished, False)

Rows after deduplication:  2000
  From CSV:                2000
  From API only:           0

Columns: ['claim_id', 'policy_id', 'loss_date', 'report_date', 'claim_amount', 'claim_status', 'cause_of_loss', '_bronze_batch_id', '_source_system']


In [5]:
valid_policy_list = list(valid_policy_ids)

reject_reasons = (
    deduped_df
    # Rule 1: claim_id must not be null
    .withColumn(
        "r_null_claim_id",
        when(col("claim_id").isNull(), "null claim_id").otherwise(None)
    )
    # Rule 2: policy_id FK must exist in silver_policies
    .withColumn(
        "r_bad_policy",
        when(
            col("policy_id").isNull() | ~col("policy_id").isin(valid_policy_list),
            "invalid policy_id"
        ).otherwise(None)
    )
    # Rule 3: claim_amount must be positive
    .withColumn(
        "r_bad_amount",
        when(
            col("claim_amount").isNull() | (col("claim_amount") <= 0),
            "invalid claim_amount"
        ).otherwise(None)
    )
    # Rule 4: claim_status must be a known value
    .withColumn(
        "r_bad_status",
        when(
            ~col("claim_status").isin(list(VALID_CLAIM_STATUSES)),
            "invalid claim_status"
        ).otherwise(None)
    )
    # Rule 5: loss_date must not be null
    .withColumn(
        "r_null_loss_date",
        when(col("loss_date").isNull(), "null loss_date").otherwise(None)
    )
    .withColumn(
        "reject_reason",
        concat_ws(" | ",
            col("r_null_claim_id"), col("r_bad_policy"),
            col("r_bad_amount"), col("r_bad_status"),
            col("r_null_loss_date")
        )
    )
    .drop("r_null_claim_id", "r_bad_policy", "r_bad_amount",
          "r_bad_status", "r_null_loss_date")
)

rejects_df = reject_reasons.filter(col("reject_reason") != "")
clean_df   = reject_reasons.filter(col("reject_reason") == "")

reject_count = rejects_df.count()
clean_count  = clean_df.count()
total_count  = deduped_df.count()

print(f"Total rows:    {total_count}")
print(f"Clean rows:    {clean_count}")
print(f"Rejected rows: {reject_count}")
print(f"Reject rate:   {reject_count/total_count*100:.1f}%")
print()

assert clean_count + reject_count == total_count, (
    f"Row count mismatch: clean={clean_count} + rejects={reject_count} != total={total_count}"
)
print("Row count assertion passed.")
print()

if reject_count > 0:
    print("Reject breakdown:")
    rejects_df.groupBy("reject_reason").count().orderBy("count", ascending=False).show(truncate=False)
else:
    print("No rejects — all rows passed validation.")

StatementMeta(, 8081f143-5a0a-4e32-8063-b016b7eb9547, 7, Finished, Available, Finished, False)

Total rows:    2000
Clean rows:    1977
Rejected rows: 23
Reject rate:   1.1%

Row count assertion passed.

Reject breakdown:
+-----------------+-----+
|reject_reason    |count|
+-----------------+-----+
|invalid policy_id|23   |
+-----------------+-----+



In [6]:
silver_df = (
    clean_df
    .drop("reject_reason")
    .withColumn("_silver_load_timestamp", current_timestamp())
    .withColumn("_silver_batch_id", lit(SILVER_BATCH_ID))
)

rejects_silver_df = (
    rejects_df
    .withColumn("_silver_load_timestamp", current_timestamp())
    .withColumn("_silver_batch_id", lit(SILVER_BATCH_ID))
)

(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE)
)
print(f"Wrote {silver_df.count()} rows to {TARGET_TABLE}")

(
    rejects_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(REJECTS_TABLE)
)
print(f"Wrote {rejects_silver_df.count()} rows to {REJECTS_TABLE}")

StatementMeta(, 8081f143-5a0a-4e32-8063-b016b7eb9547, 8, Finished, Available, Finished, False)

Wrote 1977 rows to silver_claims
Wrote 23 rows to silver_claims_rejects


In [7]:
verify_clean   = spark.read.table(TARGET_TABLE)
verify_rejects = spark.read.table(REJECTS_TABLE)

clean_count   = verify_clean.count()
reject_count  = verify_rejects.count()
total_deduped = deduped_df.count()

print(f"silver_claims verified:")
print(f"  Clean rows:    {clean_count}")
print(f"  Rejected rows: {reject_count}")
print(f"  Total deduped: {total_deduped}")
print(f"  Math checks:   {clean_count + reject_count} == {total_deduped}: {clean_count + reject_count == total_deduped}")
print()

required_lineage = ["_bronze_batch_id", "_silver_load_timestamp", "_silver_batch_id"]
has_lineage = all(c in verify_clean.columns for c in required_lineage)
print(f"  Lineage cols:  {'present' if has_lineage else 'MISSING'}")
print()

print("Reject breakdown (from rejects table):")
verify_rejects.groupBy("reject_reason").count().orderBy("count", ascending=False).show(truncate=False)

print("Sample clean rows:")
verify_clean.select(
    "claim_id", "policy_id", "claim_amount",
    "claim_status", "_source_system", "_silver_batch_id"
).show(5, truncate=False)

StatementMeta(, 8081f143-5a0a-4e32-8063-b016b7eb9547, 9, Finished, Available, Finished, False)

silver_claims verified:
  Clean rows:    1977
  Rejected rows: 23
  Total deduped: 2000
  Math checks:   2000 == 2000: True

  Lineage cols:  present

Reject breakdown (from rejects table):
+-----------------+-----+
|reject_reason    |count|
+-----------------+-----+
|invalid policy_id|23   |
+-----------------+-----+

Sample clean rows:
+-----------+----------+------------+------------+--------------+-------------------------------+
|claim_id   |policy_id |claim_amount|claim_status|_source_system|_silver_batch_id               |
+-----------+----------+------------+------------+--------------+-------------------------------+
|CLM-0000001|POL-008058|20165.26    |CLOSED      |csv           |silver_20260603_014023_db062087|
|CLM-0000002|POL-000953|11686.47    |LITIGATION  |csv           |silver_20260603_014023_db062087|
|CLM-0000003|POL-005975|648.93      |OPEN        |csv           |silver_20260603_014023_db062087|
|CLM-0000004|POL-005752|17578.86    |CLOSED      |csv           |silver_